In [0]:
df_silver_members = spark.read.table("hdfc_ergo.hdfc_silver.dim_members")
df_gold_geography = spark.read.table("hdfc_ergo.hdfc_gold.dim_geography")

In [0]:
from pyspark.sql.functions import coalesce, lit, concat, substring, current_timestamp, row_number, when, col
from pyspark.sql.window import Window

# 1. PII Masking (Using substring for maximum PySpark compatibility)
# name_masked: First letter + "***"
df_gold = df_silver_members.withColumn("name_masked", concat(substring(col("full_name"), 1, 1), lit("***"))) \

# aadhar_masked: "XXXX-XXXX-" + last 4 digits
df_gold = df_gold.withColumn("aadhar_masked", when(col("aadhar_number").isNotNull(), concat(lit("XXXX-XXXX-"), substring(col("aadhar_number"), -4, 4))).otherwise(None)) \

# pan_masked: First 3 letters + "XXXXXXXXXX"
df_gold = df_gold.withColumn("pan_masked", when(col("pan_number").isNotNull(), concat(substring(col("pan_number"), 1, 3), lit("XXXXXXXXXX"))).otherwise(None))

# 2. Lookup geography_sk FROM Dim_Geography (Triple join for precision)
df_geo_lookup = df_gold_geography.select("geography_sk", "state", "city", "pincode").distinct()

df_gold = df_gold.join(
    df_geo_lookup,
    on=["state", "city", "pincode"],
    how="left"
)

# 3. Orphan Handling for Geography (If no match, assign -1)
# CRITICAL: The -1 MUST be wrapped in lit(), otherwise Spark throws the "got int" error!
df_gold = df_gold.withColumn("geography_sk", coalesce(col("geography_sk"), lit(-1)))

# 4. Generate Surrogate Key: AUTO_INCREMENT ordered by member_id
window_spec = Window.orderBy("member_id")
df_gold = df_gold.withColumn("member_sk", row_number().over(window_spec))

# 5. Add SCD Type 2 Attributes
df_gold = df_gold.withColumn("_valid_from", current_timestamp()) \
                 .withColumn("_valid_to", lit(None).cast("timestamp")) \
                 .withColumn("_is_current", lit(True).cast("boolean"))

# 6. Select ONLY the columns required in Gold (Column Pruning)
df_gold = df_gold.select(
    "member_sk",
    "member_id",
    "name_masked",
    "aadhar_masked",
    "pan_masked",
    "geography_sk",          # Replaces city, state, pincode
    "policy_number",
    "gender",
    "dob",
    "age",
    "email",
    "phone",
    "employment_status",
    "enrollment_date",
    "termination_date",
    "member_status",
    "sum_insured",
    "deductible_amount",
    "co_pay_percentage",
    "room_rent_limit",
    "chronic_conditions",
    "tobacco_user",
    "disability_status",
    "language_preference",
    "contact_preference",
    "risk_score",
    "claims_history_count",
    "last_claims_date",
    "_valid_from",
    "_valid_to",
    "_is_current"
)

display(df_gold.limit(5))

In [0]:
gold_table_fqn = "hdfc_ergo.hdfc_gold.dim_members"

df_gold.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(gold_table_fqn)

print(f"✅ Successfully wrote Dim_Members to Gold layer.")